## 실자료 평균. 서울, 대구 일조량

In [75]:
# One sample exercise
# 평균 추론. muhat, se, 신뢰구간, 검정통계량, pvalue.

import os
import numpy as np                          # numpy 라이브러리 전체.
import pandas as pd                         # pandas 라이브러리 전체, 시계열(기본) 여기 있네.
import scipy as sci
from scipy import stats, optimize, linalg   # scipy 라이브러리 하부 모듈. 필요한 통계 모듈만.
import statsmodels.api as sm                # 방대한 모델이라 개발자들이 많이 쓰는 모델 묶음.
from statsmodels.tsa import stattools       # time series analysis
from statsmodels.tsa.arima.model import ARIMA # 이거 가능하다 이거지. 이거 대문자네. 꼭. 소문자 못읽어.
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportion_confint, proportions_ztest
from statsmodels.stats.weightstats import DescrStatsW, ztest
from scipy.stats import ttest_1samp


### 자료셋 정리
 height

In [76]:
# 1. 데이터 준비. 읽기. 생성.
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/ts_sun_sel_dg.xlsx?raw=true'
df_dat = pd.read_excel(dat_url)
df_dat #.head()


,yymm,sel,dgu
0,1979.01,115.0,180.0
1,1979.02,114.0,188.0
2,1979.03,186.0,220.0
3,1979.04,193.0,222.0
4,1979.05,249.0,245.0
...,...,...,...
367,2009. 08,150.9,NaN
368,2009. 09,201.6,NaN
369,2009. 10,236.3,NaN
370,2009. 11,130.1,NaN


### 변수 생성, 리네임 등

In [77]:
# sel, dgu 갯수 차이, 뒤에 NaN 나오네.
sun_sl = df_dat['sel']
sun_dg = df_dat['dgu']
sun_h = sun_sl          # sunny hour, choose city
sun_h = sun_h.dropna()  # NaN을 처음부터 제거하는 방법임.
n_obs = len(sun_h)
print(n_obs)

372


### 표본평균의 분포
\begin{align}
 \hat \mu & \sim N \left(\mu, SE^2 \right) \\
 \widehat{SE} & = \sqrt{ \sigma^2 \over n }
\end{align}

### 가설검정, 검정통계량
\begin{align}
 H_0  : \mu  = \mu_0 \quad vs. \quad
 H_A  : \mu  \ne 0  
\end{align}

$$
 T_0 = { \hat\mu - \mu_0 \over \widehat{ SE } }
$$



#### 유의수준, 임계치

In [78]:
# 분포 임계치, 양방향, 우측값.
alpha = 0.05                          # significance level, two side.
zcv_r = stats.norm.ppf( 1- alpha/2 )  # critical value on the right, two side
tcv_r = stats.t.ppf(1 - alpha / 2, df= n_obs-1 )
print( alpha, zcv_r, tcv_r )

0.05 1.959963984540054 1.9663788029857756


### 평균 키(표본평균) $ \hat \mu , \bar X $
$\hat \mu = { \sum X_i \over n }$

In [79]:
# 평균 키 추정치:
# 추정치: 평균, 표준오차, overall

mu_hat = sun_h.mean()
print("표본평균 추정치: " , mu_hat )


표본평균 추정치:  171.91075268817204


### 평균 키(표본평균)의 분산, 표준오차
$ V = SE^2 $, $ \ SE = \sqrt{ \sigma^2 \over n } $

In [80]:

sig2_hat = sun_h.var(ddof=1)
print("표본분산 추정치: " , sig2_hat )

sig_hat = ( sig2_hat  )**0.5
print("표준편차 추정치: " , sig_hat )

var_mu_hat = sig2_hat / n_obs
print(" 표본평균의 분산 추정치: " , var_mu_hat )

#se_muhat = var_muhat**0.5
se_mu_hat = sun_h.std(ddof=1) / np.sqrt( n_obs )
print(" 표본평균의 표준오차 추정치: " , se_mu_hat )


표본분산 추정치:  1936.772606440019
표준편차 추정치:  44.00877874288287
 표본평균의 분산 추정치:  5.206377974301127
 표본평균의 표준오차 추정치:  2.2817488850224352


### 평균추론 1. 신뢰구간
모평균의  
$ 100(1-\alpha) \% $ 신뢰구간
\begin{align}  
\hat \mu  & \pm z_{\alpha/2} \times \widehat{ SE }  \\  
\hat \mu  & \pm t_{\alpha/2, df} \times \widehat{ SE }  
\end{align}


In [81]:
# 신뢰구간, right, left,
ci_r = mu_hat + zcv_r * se_mu_hat
ci_l = mu_hat - zcv_r * se_mu_hat

print( f"   mu_hat,        se_mu_hat,              {100*(1-alpha)}%   confidence interval  ")
print(  mu_hat, se_mu_hat,   ci_l, ci_r )

print(" t분포를 적용하면 약간 차이 남")
ci_tr = mu_hat + tcv_r * se_mu_hat
ci_tl = mu_hat - tcv_r * se_mu_hat
print(  mu_hat, se_mu_hat,   ci_tl, ci_tr )

   mu_hat,        se_mu_hat,              95.0%   confidence interval  
171.91075268817204 2.2817488850224352 167.43860705176365 176.38289832458042
 t분포를 적용하면 약간 차이 남
171.91075268817204 2.2817488850224352 167.4239700469275 176.39753532941657


### 평균추론 2. 검정통계량, $ T_0 $
$$
 T_0 = { \hat\mu - \mu_0 \over \widehat{ SE } }
$$


#### 귀무가설, 대립가설
$ H_0 : \mu = \mu_0 $ vs. $ H_A: \mu \ne \mu_0 $

\begin{align}
 \text{reject } H_0
 & \quad
 \text{ if } |T_0| > z_{\alpha/2} \\
 & \quad
 \text{ if  p-value } < \alpha
\end{align}

In [82]:
# 가설검정. 전체, 톨
# 귀무가설 H0: mu = mu0
# 검정통계치, pvalue

mu_zero = 190
print("H0: mu = ", mu_zero, "HA: mu is not ", mu_zero)


H0: mu =  190 HA: mu is not  190


#### 검정통계치, 가설검정

In [83]:
t_0 = np.abs( ( mu_hat - mu_zero ) / se_mu_hat )
print("t_0= ", t_0 )
print( f"reject H0 if test statistic, t_0 = {t_0} > critical value zcv_r = {zcv_r} at significance level {alpha}" )
result_test = " 'Reject H0' " if t_0 > zcv_r else " 'Fail to reject H0' "   # 이건 되는 군.
print(f" 검정통계치,    임계치(유의수준={alpha}),    검정결과 ")
print( t_0 , zcv_r, result_test)

t_0=  7.9277993431122455
reject H0 if test statistic, t_0 = 7.9277993431122455 > critical value zcv_r = 1.959963984540054 at significance level 0.05
 검정통계치,    임계치(유의수준=0.05),    검정결과 
7.9277993431122455 1.959963984540054  'Reject H0' 


### 평균추론 3. $ p $값
$ p$ value $= 2\times P(T_0 > |t_0|) $

In [84]:
# 이제 p값

p_val = 2*( 1 - stats.norm.cdf( t_0 ) )
p_val_t = 2*( 1 - stats.t.cdf( t_0, df= n_obs -1 ) )
p_val_sf = 2 *  stats.t.sf( abs( t_0 ), df= n_obs - 1)

print(f" p값 = { p_val } " )
print(f" p값 = { p_val_t } " )
print(f" p값 = { p_val_sf } " )


f"reject H0 if p값 < {alpha}"


 p값 = 2.220446049250313e-15 
 p값 = 2.6423307986078726e-14 
 p값 = 2.6437456327902076e-14 


'reject H0 if p값 < 0.05'

### 모듈로 확인, confirm

In [89]:
# # 모듈 이용 statsmodels.stats.weightstats, 다른 모듈 없나???
# 차이가 나. 자유도 조정 안 하는 모양.
#    from statsmodels.stats.weightstats import DescrStatsW
ds_sun_h = DescrStatsW( sun_h )              # 모듈이 돌려준 객체
ci_ds = ds_sun_h.tconfint_mean(alpha=0.05)   # 이건 tuple

print(" 모듈 DescrStatsW 반환 값; 자유도 조정 안 하는군.  ")
print(f" 표본평균 : {ds_sun_h.mean:.8f}")
print(f" 표본분산 : {ds_sun_h.var:.8f}")
print(f" 표준편차 : {ds_sun_h.std:.8f}")
print(f" 표준오차 : {(ds_sun_h.std/np.sqrt(n_obs )):.8f}")
print(" 신뢰구간 :", ", ".join(f"{x:.8f}" for x in ci_ds ))

print("# 차이가 나. DescrStatsW 자유도 조정 방식 ")
print("# var, n. std, n-1. 방식이 다르다고 하네.")
print("# 자유도 조정 후 계산값")
var_df = ds_sun_h.var * ( n_obs / ( n_obs - 1 ) )
std_df = var_df**0.5
se_df = std_df / np.sqrt( n_obs )
print(f"# 표본분산 : {var_df:.8f} ")
print(f"# 표준편차 :  {std_df:.8f} ")
print(f"# 표준오차 :  {se_df:.8f}")

#print(" 신뢰구간 :", ", ".join(f"{x:.8f}" for x in ci_a ))

# 평균 t 검정. 모듈이 여럿임.
print(" 모듈 DescrStatsW 반환, ttest_mean() 적용 ")
print(" H0: mu =", mu_zero )
t0_ds, pval_ds, df_ds = ds_sun_h.ttest_mean(mu_zero) # mu_zero겠지?

print(" 검정통계치          " , t0_ds)
print(" 자유도(여기서는 조정)" , df_ds )
print(" p-value  ",  pval_ds  )
# pvalu의 미세한 차이는 t.cdf()와 t.sf() 차이네.

 모듈 DescrStatsW 반환 값; 자유도 조정 안 하는군.  
 표본평균 : 171.91075269
 표본분산 : 1931.56622847
 표준편차 : 43.94958735
 표준오차 : 2.27867995
 신뢰구간 : 167.42397005, 176.39753533
# 차이가 나. DescrStatsW 자유도 조정 방식 
# var, n. std, n-1. 방식이 다르다고 하네.
# 자유도 조정 후 계산값
# 표본분산 : 1936.77260644 
# 표준편차 :  44.00877874 
# 표준오차 :  2.28174889
 모듈 DescrStatsW 반환, ttest_mean() 적용 
 H0: mu = 190
 검정통계치           -7.9277993431122455
 자유도(여기서는 조정) 371.0
 p-value   2.6437456327902076e-14


## 2개 그룹 평균 비교는 별도